# Curriculum 03 · Lab 4 — metadata filtering + index/query/memory benchmark

**Goal:** Add the two things real apps need on top of the store basics from
labs 01–03: **metadata filtering** (scope retrieval to a subset like "only
bucket b1") and a small **benchmark** (index time, query latency, RAM, disk)
so you can see why you'd pick one store over another.

```
Filtering   : plain dict {"bucket": "b1"} on all three stores
Qdrant form : real models.Filter with a metadata. key — two silent traps
FAISS quirk : filters in Python over a fetch_k=20 window -> may return < top_k
Benchmark   : index time | query ms | RSS +MB | disk bytes (per store)
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia (100 passages, synthetic buckets b0..b3)
```

**Why this lab exists:** labs 01–03 proved the stores rank identically; this
lab stops treating them as interchangeable and measures *how* they differ.
The filter section is where the Qdrant `metadata.` prefix and the FAISS
`fetch_k` window actually bite — both silent (bad filters return 0 hits, no
error), both demonstrated with live queries before the gate locks the
correct forms in.

This is the last lab of track 03-vector-databases (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's vector-store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`src/vectordb/*.py` and `src/embeddings/bge.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The store classes live in the repo's shared
component library (`src/vectordb/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   faiss-cpu, chromadb, qdrant-client -> the three stores under test
#   psutil                -> process RSS (memory benchmark)
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu chromadb "qdrant-client==1.13.3" psutil pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import gc
import shutil
import sys
import tempfile
import time
from pathlib import Path

import pandas as pd
import psutil

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from qdrant_client.http import models  # noqa: E402
from vectordb.chroma import ChromaVectorStore  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402
from vectordb.qdrant import QdrantVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** The usual corpus/embedding constants plus the lab's own knobs:
`BUCKETS` (the four synthetic metadata values, cycled over passages),
`FILTER_BUCKET` (which bucket every filtered query asks for), `QUERY_REPEATS`
(how many repeated queries the latency average is taken over), and
`FAISS_FETCH_K` (langchain-FAISS's candidate window for filtered searches).

**WHY:** The bucket tags are synthetic — attached by index, not by content —
which is why the content check at the end runs against the *unfiltered*
ranking: only the unfiltered top-1 is guaranteed to carry the answer.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus
QUESTION_IDS = [1606, 1610]  # real questions; answers live inside the subset
TOP_K = 3
PREVIEW = 62
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
BUCKETS = ["b0", "b1", "b2", "b3"]  # synthetic metadata: 4 buckets, cycled
FILTER_BUCKET = "b1"
QUERY_REPEATS = 20  # per question, for the latency average
FAISS_FETCH_K = 20  # langchain-FAISS default candidate window for filters
COLLECTION = "lab04"


## 2 · Load — corpus + questions + benchmark helpers

**WHAT:** The corpus/format helpers from labs 01–03, plus three
benchmark-specific ones: `rss_mb` (current process RSS in MiB via psutil),
`dir_bytes` (total bytes of every file under a path — 0 if it doesn't
exist), and `qdrant_filter` (the CORRECT expressive Qdrant filter with the
`metadata.`-prefixed key).

**WHY:** `rss_mb` and `dir_bytes` are how the benchmark answers "how much
RAM" and "how much disk" per store without any external tooling.
`qdrant_filter` is the one right way to build an expressive Qdrant filter —
the lab's traps are the *wrong* ways, demonstrated with live queries in the
experiment.


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def rss_mb() -> float:
    """Current process RSS in MiB (Linux/psutil)."""
    return psutil.Process().memory_info().rss / (1024 * 1024)


def dir_bytes(path: Path) -> int:
    """Total size of every file under ``path`` (0 if it doesn't exist)."""
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())


def qdrant_filter(bucket: str) -> models.Filter:
    """The CORRECT expressive Qdrant filter: ``metadata.``-prefixed key."""
    return models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.bucket", match=models.MatchValue(value=bucket)
            )
        ]
    )


## 3 · Experiment — embed once, tag with bucket metadata, filter and benchmark

**WHAT:** `build_chunks` attaches the synthetic `bucket` tag (b0..b3 cycled
by passage index). `run_experiment` embeds once, indexes the same vectors
into all three stores, runs every filter form against each store — including
the two Qdrant traps and the FAISS fetch-window behavior — and benchmarks
index time, query latency, RSS delta, and disk footprint per store.

**WHY:** Filtering is tested exhaustively because the failures are silent: a
bad Qdrant filter returns 0 hits with no error. The benchmark numbers are
the payoff of labs 01–03 — same vectors, same queries, three very different
resource profiles.


In [5]:
def build_chunks(
    passage_texts: list[str], passage_ids: list[int]
) -> list[Document]:
    """Attach synthetic ``bucket`` metadata: b0..b3 cycled by passage index."""
    return [
        Document(
            page_content=t,
            metadata={"id": pid, "bucket": BUCKETS[i % len(BUCKETS)]},
        )
        for i, (t, pid) in enumerate(zip(passage_texts, passage_ids))
    ]


def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    chunks = build_chunks(passage_texts, passage_ids)

    # --- Embed the subset once; ALL THREE stores index the same vectors ----
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0
    query_vecs = [embedder.embed_query(q) for _, q in questions]

    # --- Memory checkpoints (RSS is process-wide; deltas are cumulative) ---
    gc.collect()
    baseline_rss = rss_mb()

    # --- FAISS — in-memory, zero disk --------------------------------------
    faiss_store = FAISSVectorStore()
    t0 = time.perf_counter()
    faiss_store.add(chunks, embeddings=passage_vecs)
    faiss_add_s = time.perf_counter() - t0
    gc.collect()
    faiss_rss = rss_mb() - baseline_rss

    # --- Chroma — persistent in a temp directory ---------------------------
    chroma_dir = Path(tempfile.mkdtemp(prefix="lab04_chroma_"))
    chroma_store = ChromaVectorStore(
        collection_name=COLLECTION, persist_dir=str(chroma_dir)
    )
    t0 = time.perf_counter()
    chroma_store.add(chunks, embeddings=passage_vecs)
    chroma_add_s = time.perf_counter() - t0
    gc.collect()
    chroma_rss = rss_mb() - baseline_rss

    # --- Qdrant — in-memory (":memory:" writes nothing to disk) ------------
    qdrant_store = QdrantVectorStore(collection_name=COLLECTION, path=":memory:")
    t0 = time.perf_counter()
    qdrant_store.add(chunks, embeddings=passage_vecs)
    qdrant_add_s = time.perf_counter() - t0
    gc.collect()
    qdrant_rss = rss_mb() - baseline_rss

    # --- Filtered retrieval (same logical filter, three syntaxes) ----------
    faiss_filtered = [
        faiss_store.query_with_scores(q, top_k=TOP_K, filter={"bucket": FILTER_BUCKET})
        for q in query_vecs
    ]
    chroma_filtered = [
        chroma_store.query_with_scores(q, top_k=TOP_K, filter={"bucket": FILTER_BUCKET})
        for q in query_vecs
    ]
    qdrant_simple = [
        qdrant_store.query_with_scores(q, top_k=TOP_K, filter={"bucket": FILTER_BUCKET})
        for q in query_vecs
    ]
    # Expressive form: must be a real models.Filter with a metadata.-prefixed
    # key — the raw-dict "must" and the unprefixed key both silently return 0.
    qdrant_model = [
        qdrant_store.query_with_scores(q, top_k=TOP_K, filter=qdrant_filter(FILTER_BUCKET))
        for q in query_vecs
    ]
    qdrant_must_dict = [
        qdrant_store.query_with_scores(
            q,
            top_k=TOP_K,
            filter={"must": [{"key": "bucket", "match": {"value": FILTER_BUCKET}}]},
        )
        for q in query_vecs
    ]
    qdrant_unprefixed = [
        qdrant_store.query_with_scores(
            q,
            top_k=TOP_K,
            filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="bucket", match=models.MatchValue(value=FILTER_BUCKET)
                    )
                ]
            ),
        )
        for q in query_vecs
    ]

    # Unfiltered baseline: content grounding (the bucket tags are synthetic,
    # so only the unfiltered top-1 is guaranteed to carry the answer).
    faiss_unfiltered = [
        faiss_store.query_with_scores(q, top_k=TOP_K) for q in query_vecs
    ]

    # --- Benchmark: query latency (mean over repeats x questions) ----------
    def latency_ms(store, scored_unfiltered_call) -> float:
        total = 0.0
        n = 0
        for _ in range(QUERY_REPEATS):
            for q in query_vecs:
                t0 = time.perf_counter()
                scored_unfiltered_call(store, q)
                total += time.perf_counter() - t0
                n += 1
        return total / n * 1000.0

    faiss_lat = latency_ms(faiss_store, lambda s, q: s.query_with_scores(q, top_k=TOP_K))
    chroma_lat = latency_ms(chroma_store, lambda s, q: s.query_with_scores(q, top_k=TOP_K))
    qdrant_lat = latency_ms(qdrant_store, lambda s, q: s.query_with_scores(q, top_k=TOP_K))

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
        "stores": {
            "faiss": {
                "add_s": faiss_add_s, "rss_mb": faiss_rss,
                "disk_bytes": 0, "latency_ms": faiss_lat,
            },
            "chroma": {
                "add_s": chroma_add_s, "rss_mb": chroma_rss,
                "disk_bytes": dir_bytes(chroma_dir), "latency_ms": chroma_lat,
            },
            "qdrant": {
                "add_s": qdrant_add_s, "rss_mb": qdrant_rss,
                "disk_bytes": 0, "latency_ms": qdrant_lat,
            },
        },
        "faiss_filtered": faiss_filtered,
        "chroma_filtered": chroma_filtered,
        "qdrant_simple": qdrant_simple,
        "qdrant_model": qdrant_model,
        "qdrant_must_dict": qdrant_must_dict,
        "qdrant_unprefixed": qdrant_unprefixed,
        "faiss_unfiltered": faiss_unfiltered,
        "chroma_dir": chroma_dir,
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — embedding, three index builds, all six
filter forms, and the latency benchmark take a few seconds. The artifact
dict is kept as `exp`.

**WHY:** As in the earlier labs, demo and gate both read this single `exp`.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the benchmark table (add time, RSS delta, disk
bytes, query ms per store), the filtered top-3 for Q1610 across all filter
forms — including the two Qdrant traps returning `[]` and FAISS possibly
returning fewer than 3 — and the takeaway.

**WHY:** Read the table like a trade-off: FAISS builds in milliseconds and
holds everything in RAM (0 bytes on disk); Chroma is slower to build but
persists to a directory; Qdrant `:memory:` is mid-pack with a full filter
language. The filter block is the trap showcase — the silent `[]` results
are the whole lesson.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 04 — metadata filtering + index/query/memory benchmark")
    print(f"{BGE_MODEL_NAME} | 100 passages | synthetic buckets {BUCKETS}")
    print("=" * 66)

    print(f"\n[1] Corpus + embedding:")
    print(f"    {exp['indexed']} passages, dim {exp['dim']}, embedded in {exp['embed_s']:.2f}s")
    print(f"    each passage tagged bucket={BUCKETS}, cycled by index")

    print(f"\n[2] Index build — time, memory (RSS delta vs post-embedding), disk:")
    print(f"    {'store':<8}{'add time':>10}{'rss +MB':>10}{'disk':>10}{'query ms':>10}")
    for name, s in exp["stores"].items():
        print(f"    {name:<8}{s['add_s']:>9.3f}s{s['rss_mb']:>10.1f}"
              f"{s['disk_bytes']:>9}B{s['latency_ms']:>9.3f}")

    print(f"\n[3] Metadata filter — top-{TOP_K} within bucket '{FILTER_BUCKET}':")
    qid, qtext = exp["questions"][1]
    print(f'    Q[{qid}] "{qtext}"')
    faiss_ids = [d.metadata["id"] for d, _ in exp["faiss_filtered"][1]]
    chroma_ids = [d.metadata["id"] for d, _ in exp["chroma_filtered"][1]]
    qdrant_simple_ids = [d.metadata["id"] for d, _ in exp["qdrant_simple"][1]]
    qdrant_model_ids = [d.metadata["id"] for d, _ in exp["qdrant_model"][1]]
    print(f"    FAISS   {{'bucket': '{FILTER_BUCKET}'}} -> {faiss_ids}")
    if len(faiss_ids) < TOP_K:
        print(f"    (FAISS found only {len(faiss_ids)} of {TOP_K} — its filter is a Python")
        print(f"     post-filter over the top {FAISS_FETCH_K} candidates; Chroma/Qdrant")
        print(f"     filter in-engine and always return {TOP_K})")
    print(f"    Chroma  {{'bucket': '{FILTER_BUCKET}'}} -> {chroma_ids}")
    print(f"    Qdrant  {{'bucket': '{FILTER_BUCKET}'}} -> {qdrant_simple_ids}")
    print(f"    Qdrant  models.Filter(metadata.bucket) -> {qdrant_model_ids}")
    print(f"    Qdrant  raw {{'must': [...]}} dict     -> "
          f"{[d.metadata['id'] for d, _ in exp['qdrant_must_dict'][1]]}  (silent trap)")
    print(f"    Qdrant  models.Filter(bucket, no meta.) -> "
          f"{[d.metadata['id'] for d, _ in exp['qdrant_unprefixed'][1]]}  (silent trap)")

    print("\n[4] Takeaway")
    print("    A filter narrows the candidate set; ranking inside it still")
    print("    follows the same similarity order in every store. The plain")
    print("    dict works everywhere, but Qdrant's expressive form needs a")
    print("    real models.Filter with the metadata. prefix — and bad forms")
    print("    return 0 hits instead of erroring. FAISS may return fewer")
    print("    than top_k: its filter is a Python post-filter over a")
    print(f"    fetch_k={FAISS_FETCH_K} candidate window, while Chroma/Qdrant")
    print("    filter in the engine. Benchmark: FAISS and Qdrant(:memory:)")
    print("    hold everything in RAM (0 bytes on disk); Chroma trades a")
    print("    little memory for a persistent sqlite dir. RSS deltas are")
    print("    process-wide and noisy — compare roughly.")
    print(f"    (temp chroma dir {exp['chroma_dir']} removed after this run)")


In [8]:
print_demo(exp)


Lab 04 — metadata filtering + index/query/memory benchmark
BAAI/bge-base-en-v1.5 | 100 passages | synthetic buckets ['b0', 'b1', 'b2', 'b3']

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 14.58s
    each passage tagged bucket=['b0', 'b1', 'b2', 'b3'], cycled by index

[2] Index build — time, memory (RSS delta vs post-embedding), disk:
    store     add time   rss +MB      disk  query ms
    faiss       0.045s      12.2        0B    0.043
    chroma      2.942s      61.4  1255588B    1.119
    qdrant      0.082s      62.5        0B    0.271

[3] Metadata filter — top-3 within bucket 'b1':
    Q[1610] "Who founded Montevideo?"
    FAISS   {'bucket': 'b1'} -> [5, 13]
    (FAISS found only 2 of 3 — its filter is a Python
     post-filter over the top 20 candidates; Chroma/Qdrant
     filter in-engine and always return 3)
    Chroma  {'bucket': 'b1'} -> [5, 13, 53]
    Qdrant  {'bucket': 'b1'} -> [5, 13, 53]
    Qdrant  models.Filter(metadata.bucket) -> [5, 13, 53]
    Qdra

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: dimension/count, FAISS's filtered
hits being the top-len(FAISS) of the engine-filtered ranking, all hits in
the requested bucket, the FAISS-fetch-window vs engine-count asymmetry, the
correct Qdrant form matching the dict form, both traps returning 0 hits, the
content check against the unfiltered ranking, and benchmark sanity.

**WHY:** `python 04-benchmark.py --verify` must print 10/10 PASS; this cell
proves the notebook reproduces the verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Unfiltered ranking identity across the three stores (labs 01–03 carryover).
    def ids(scored) -> list[int]:
        return [d.metadata["id"] for d, _ in scored]

    all_same_rank = True
    for i in range(len(exp["questions"])):
        f = ids(exp["faiss_filtered"][i])
        c = ids(exp["chroma_filtered"][i])
        q = ids(exp["qdrant_simple"][i])
        # FAISS filters inside a fetch_k-wide candidate window, so it may
        # return FEWER than TOP_K hits; its hits are still the top-len(f)
        # passages of the engine-filtered ranking, in the same order.
        all_same_rank &= f == c[: len(f)] == q[: len(f)]
    checks.append(("FAISS filtered hits are the top-len(FAISS) of the engine-filtered ranking", all_same_rank))

    # The core claim: filtered hits are exactly the requested bucket. FAISS
    # may return fewer than TOP_K (Python post-filter over fetch_k candidates);
    # Chroma/Qdrant filter in-engine and always return exactly TOP_K.
    only_b1 = True
    faiss_count_ok = True
    engine_count_ok = True
    for i in range(len(exp["questions"])):
        for key, is_faiss in (
            ("faiss_filtered", True),
            ("chroma_filtered", False),
            ("qdrant_simple", False),
        ):
            hits = exp[key][i]
            only_b1 &= all(d.metadata["bucket"] == FILTER_BUCKET for d, _ in hits)
            if is_faiss:
                faiss_count_ok &= 0 < len(hits) <= TOP_K
            else:
                engine_count_ok &= len(hits) == TOP_K
    checks.append(("every store returns only bucket-b1 hits", only_b1))
    checks.append(("FAISS may return < top_k (fetch-window filter); Chroma/Qdrant always top_k", faiss_count_ok and engine_count_ok))

    # The expressive Qdrant forms: the correct one matches the dict form...
    q_model_ids = [ids(exp["qdrant_model"][i]) for i in range(len(exp["questions"]))]
    q_simple_ids = [ids(exp["qdrant_simple"][i]) for i in range(len(exp["questions"]))]
    checks.append(("Qdrant models.Filter (metadata.bucket) matches the dict form", q_model_ids == q_simple_ids))

    # ...and the two silent traps return ZERO hits (no error, no results).
    traps_empty = all(len(exp["qdrant_must_dict"][i]) == 0 for i in range(len(exp["questions"])))
    checks.append(("Qdrant raw {'must': [...]} dict returns 0 hits (documented trap)", traps_empty))
    unprefixed_empty = all(len(exp["qdrant_unprefixed"][i]) == 0 for i in range(len(exp["questions"])))
    checks.append(("Qdrant models.Filter without metadata. prefix returns 0 hits (trap)", unprefixed_empty))

    # Content check (same as labs 01–03) — against the UNFILTERED ranking:
    # the bucket tags are synthetic, so only the unfiltered top-1 is
    # guaranteed to carry the answer.
    q1610_top = exp["faiss_unfiltered"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))

    # Benchmark sanity: measurable timings, non-negative memory, disk story.
    bench_ok = True
    for name, s in exp["stores"].items():
        bench_ok &= s["add_s"] > 0
        bench_ok &= s["latency_ms"] > 0
        bench_ok &= s["rss_mb"] >= 0
        if name == "chroma":
            bench_ok &= s["disk_bytes"] > 0
        else:
            bench_ok &= s["disk_bytes"] == 0
    checks.append(("benchmark: timings positive, Chroma on disk, FAISS/Qdrant RAM-only", bench_ok))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


if __name__ == "__main__":
    exp = run_experiment()
    try:
        if "--verify" in sys.argv:
            sys.exit(verify_gate(exp))
        print_demo(exp)
    finally:
        shutil.rmtree(exp["chroma_dir"], ignore_errors=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Lab 04 — metadata filtering + index/query/memory benchmark
BAAI/bge-base-en-v1.5 | 100 passages | synthetic buckets ['b0', 'b1', 'b2', 'b3']

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 6.37s
    each passage tagged bucket=['b0', 'b1', 'b2', 'b3'], cycled by index

[2] Index build — time, memory (RSS delta vs post-embedding), disk:
    store     add time   rss +MB      disk  query ms
    faiss       0.004s       0.0        0B    0.042
    chroma      0.070s       1.7  1255588B    1.193
    qdrant      0.082s       1.7        0B    0.288

[3] Metadata filter — top-3 within bucket 'b1':
    Q[1610] "Who founded Montevideo?"
    FAISS   {'bucket': 'b1'} -> [5, 13]
    (FAISS found only 2 of 3 — its filter is a Python
     post-filter over the top 20 candidates; Chroma/Qdrant
     filter in-engine and always return 3)
    Chroma  {'bucket': 'b1'} -> [5, 13, 53]
    Qdrant  {'bucket': 'b1'} -> [5, 13, 53]
    Qdrant  models.Filter(metadata.bucket) -> [5, 13, 53]
    Qdran

In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 100 passages indexed
  [PASS] FAISS filtered hits are the top-len(FAISS) of the engine-filtered ranking
  [PASS] every store returns only bucket-b1 hits
  [PASS] FAISS may return < top_k (fetch-window filter); Chroma/Qdrant always top_k
  [PASS] Qdrant models.Filter (metadata.bucket) matches the dict form
  [PASS] Qdrant raw {'must': [...]} dict returns 0 hits (documented trap)
  [PASS] Qdrant models.Filter without metadata. prefix returns 0 hits (trap)
  [PASS] Q1610 top-1 names the Spanish founder of Montevideo
  [PASS] benchmark: timings positive, Chroma on disk, FAISS/Qdrant RAM-only


0

## 7 · Cleanup — remove the temp Chroma directory

**WHAT:** Deletes the temporary Chroma directory, the same way the lab
script's `finally` block does.

**WHY:** Regenerable artifact — no reason to keep it after the demo/gate.


In [11]:
import shutil
shutil.rmtree(exp["chroma_dir"], ignore_errors=True)
